# Happy GPT
 

## Setup 

Before you begin make sure that the given `mingpt` folder is in the same directory as the `happyGPT.ipynb` notebook. Below we will first install all the necessary libraries. 


In [ ]:
%pip install datasets torch sentiment-nltk

After installing the required packages above, please run the cell below to import the libraries we are going to use.

In [1]:
import copy
import os

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import torch
from torch.utils.data.dataloader import DataLoader

from mingpt.logger import Logger
from mingpt.model import GPT
from mingpt.trainer import Trainer
from mingpt.utils import set_seed, TweetDataset
from mingpt.rewards import ValueModel, calculate_advantage_and_returns

Please also run the cell below as we will use the following function later.

In [2]:
# Helper function for computing the validation loss and logging during training
# Do not worry about the undefined variables, we will define them later.
def batch_end_callback(trainer):
    model = trainer.model
    model.eval()

    trainer.logger.log("Train", trainer.iter_num, trainer.loss.item())
    # Compute the validation loss 
    if trainer.iter_num % trainer.config.log_every == 0:
        # evaluate both the train and test score
        with torch.no_grad():
            total_loss = 0
            for i, batch in enumerate(valid_loader):
                x, y, mask = [x.to(trainer.device) for x in batch]
                logits, loss = model(x, y, attention_mask=mask)
                total_loss += loss.item()
        # Log the validation loss
        val_loss = total_loss / (i+1)
        trainer.logger.log("Valid", trainer.iter_num, val_loss)
        print(f"E: {trainer.epoch}, iter_dt {trainer.iter_dt * 1000:.2f}ms; iter {trainer.iter_num}: train loss {trainer.loss.item():.5f}, val loss: {val_loss:.5f}")
    # Generate and log sample responses during training
    if trainer.iter_num % trainer.config.generate_every == 0:
        with torch.no_grad():
            context = train_ds.tokenizer(TEST_PROMPT)[None]
            x = context.to(trainer.device)
            y = model.generate(x, 140, top_k=30, do_sample=True)[0]
            completion = train_ds.tokenizer.decode(y)
            print(completion)

    # Revert model to training mode
    model.train()


## Pre-training happyGPT  

Run the code below to pre-train the model using the tweet sentiment extraction dataset from hugging face.

In [5]:
# It may take ~10mins to run in a 16GB RAM machine with an 8-core CPU
import torch._dynamo
torch._dynamo.config.suppress_errors = True
set_seed(424242)
print("===== STARTING PRETRAINING =====")

# Dataset and tokenizer configuration 
TEST_PROMPT = "⏎I think "
valid_iters = 32
block_size = 32
train_ds = TweetDataset(block_size, split='train')
valid_ds = TweetDataset(block_size, split='test', tokenizer=train_ds.tokenizer)
end_of_text = train_ds.tokenizer.eot_token

# Model configuration
model_config = GPT.get_default_config()
model_config.model_type = 'gpt-micro'
model_config.vocab_size = train_ds.get_vocab_size()
model_config.block_size = block_size
model = GPT(model_config)

# Training configuration
train_config = Trainer.get_default_config()
train_config.learning_rate = 1e-3
train_config.num_workers = 0
train_config.log_every = 500
train_config.generate_every = 1000
train_config.epochs = 10
train_config.compile = True
trainer = Trainer(train_config, model, train_ds)

sample_size = 3
device = trainer.device
sample_prompt = torch.full((sample_size, 1), end_of_text, dtype=torch.long, device=device)

valid_loader = DataLoader(
    valid_ds,
    shuffle=False,
    batch_size=trainer.config.batch_size * 2,
)

trainer.set_callback('on_batch_end', batch_end_callback)
trainer.run()

print("\n===== DONE PRETRAINING =====")

===== STARTING PRETRAINING =====
data has 1921544 characters, 103 unique.
number of parameters: 0.81M
running on device cpu


W0106 17:01:02.211000 34632 site-packages\torch\_dynamo\convert_frame.py:1233] WON'T CONVERT forward c:\Users\User\Git\MasterKL\NN for NLP\Exercise 4 Question\mingpt\model.py line 280 
W0106 17:01:02.211000 34632 site-packages\torch\_dynamo\convert_frame.py:1233] due to: 
W0106 17:01:02.211000 34632 site-packages\torch\_dynamo\convert_frame.py:1233] Traceback (most recent call last):
W0106 17:01:02.211000 34632 site-packages\torch\_dynamo\convert_frame.py:1233]   File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_dynamo\convert_frame.py", line 1164, in __call__
W0106 17:01:02.211000 34632 site-packages\torch\_dynamo\convert_frame.py:1233]     result = self._inner_convert(
W0106 17:01:02.211000 34632 site-packages\torch\_dynamo\convert_frame.py:1233]              ^^^^^^^^^^^^^^^^^^^^
W0106 17:01:02.211000 34632 site-packages\torch\_dynamo\convert_frame.py:1233]   File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_dynamo

E: 0, iter_dt 0.00ms; iter 0: train loss 4.62406, val loss: 4.19748
⏎I think  he7lt$U 1eJ`Uu.UytHt orU\ PtIlg lI55tGtr7HVeotÂiv'7mPU7eIugPPod  gh?uPt )dÂ t]1i)gdIr.dyo\te/d)o7y KU/IVd.he//Uui/uVa7\l Ht]ht'r .e7rI s7ee
E: 1, iter_dt 252.08ms; iter 500: train loss 2.11464, val loss: 2.08760
E: 2, iter_dt 216.07ms; iter 1000: train loss 1.88416, val loss: 1.90620
⏎I think it`s not at ertaking our with get  i weat no dont hand the perd ozve the lost⏎eatty⏎n caus. It stuf IT sall but rill it seel not to so and c
E: 3, iter_dt 174.47ms; iter 1500: train loss 1.91165, val loss: 1.81251
E: 4, iter_dt 253.17ms; iter 2000: train loss 1.79267, val loss: 1.76279
⏎I think until the cops for then posplect ill sleeps frent.. get that`s too are your savered of the suppyreans!! n.... littlad is sucker didnt puts⏎y
E: 5, iter_dt 223.54ms; iter 2500: train loss 1.86457, val loss: 1.71956
E: 7, iter_dt 334.41ms; iter 3000: train loss 1.67127, val loss: 1.69637
⏎I think the sexy maybody.⏎start off then cer

Run the cell below to generate 3 tweets using the model after pre-training.

In [6]:
idx = model.generate(sample_prompt, max_new_tokens=80, top_k=30, do_sample=True)
for j,generation in enumerate(idx):
    print(f"Generation {j}:", train_ds.tokenizer.decode(generation))


Generation 0: ⏎ A pupp...  I think im forgrated thing to lank instice to make  so much   start 
Generation 1: ⏎is sunbarhed me new today takesks win!! itrink   hope  here huhhh for staming ba
Generation 2: ⏎ awy what up⏎top..  oo but yes, the issucks gubs sing but it`s any-ipod but is l


## Reward Model
Below we define the reward model we are going to use for RL. For simplicity we will use an existing reward model that given a sentence returns a numerical score capturing the positivity in terms of sentiment of the given sentence. Fill in the code below and run the cell. 

In [9]:
class SentimentRewardModel:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.sid = SentimentIntensityAnalyzer() #arbeitet auf Strings

    def sentiment(self, sentence: str) -> float:
        return self.sid.polarity_scores(sentence)['compound']

    def __call__(self, tokens):
        # the variable tokens is a tensor containing `batch_size` tokenized sentences
        # Achtung tokenized sentences -> wir müssen ecoden um reward zu berechnen (sentiment() erwartet String)
        batch_size = tokens.shape[0]
        rewards = torch.zeros(batch_size) # one reward per generated sentence
        # TODO: for each sentence in the tensor tokens compute the predicted reward (4 points)
        # Hint: use the function sentiment above (note that it expects a string as input)
        for i in range(batch_size):
            # 1. Tokens → Text, we need to decode the tokens to get a string sentence
            sentence = self.tokenizer.decode(
                tokens[i],
            )

            # 2. Sentiment-Score berechnen
            rewards[i] = self.sentiment(sentence)

        # 3. Rewards auf dasselbe Device verschieben
        return rewards.to(tokens.device)

## Fine tune the model with RL 
Fill in the code below and run the cell to fine tune the model.

In [10]:
import nltk
nltk.download('vader_lexicon')
#
print("\n===== STARTING RL =====")

# Keep a copy of the pre-trained model as the reference model (old policy)
ref_model = copy.deepcopy(model)
ref_model.requires_grad_(False)
ref_model.eval()
# Initialize the reward model
reward_model = SentimentRewardModel(train_ds.tokenizer)

# For PPO we use a model to approximate the value function
value_model = ValueModel(model_config)


batch_size = 32
num_iters = 70
learning_rate = 3e-4
grad_norm_clip = 1.0
kl_beta = 0.1
model.to(device)
value_model.to(device)
logger = Logger()

optim_groups = [{"params": model.parameters()}, {"params": value_model.parameters()}]
optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95))
prompt = torch.full((batch_size, 1), end_of_text, dtype=torch.long, device=device)

for i in range(num_iters):
    # ------------------------------------------------------------
    # Sample Data
    # These are the `actions` in the RL sense that the model takes
    # ------------------------------------------------------------
    with torch.no_grad():
        model.eval()
        value_model.eval()

        # Sample data, i.e., generate a tweet
        completion = model.generate(prompt, max_new_tokens=block_size, top_k=30, do_sample=True)
        target = completion[:, 1:]
        completion = completion[:, :-1]
        original_log_probs = model.log_probs(completion, target)

        # Reference logprobs
        ref_log_probs = ref_model.log_probs(completion, target)

        # Use the reward model to compute the reward (positivity) of the generated tweet
        rewards = torch.zeros_like(original_log_probs)
        flat_rewards = reward_model(completion)
        rewards[:, -1] = flat_rewards

        # Calculate values, returns and advantages
        values = value_model(completion)

        kl = original_log_probs - ref_log_probs
        score = rewards - kl_beta * kl

        advantages, returns = calculate_advantage_and_returns(score, values, gamma=1.0, lambd=0.95)

    # -------------
    # Train via PPO
    # -------------
    model.train()
    value_model.train()
    # Forward pass through the latest model
    log_probs = model.log_probs(completion, target)

    # Policy loss
    # TODO: compute the ppo ratio (probability ratio between the new and the old policy) (2 points)
    # Hint use the variables log_probs and original_log_probs
    
    # log π_θ(a|s) − log π_θ_old(a|s)
    logratio = log_probs - original_log_probs
    
    # π_θ(a|s) / π_θ_old(a|s)
    ppo_ratio = torch.exp(logratio)

    pg_loss1 = -advantages * ppo_ratio
    
    #Gradient clipping
    pg_loss2 = -advantages * torch.clamp(ppo_ratio, 0.8, 1.2)
    pg_loss = torch.max(pg_loss1, pg_loss2).mean()

    # Value loss
    # TODO: compute the predicted value of the generated text using the value model (1 point)
    predicted_value = value_model(completion)
    # TODO: compute the mean square error between the variables `returns` and  `predicted_value` (1 point)
    v_loss = torch.mean((returns - predicted_value) ** 2)

    loss = pg_loss + 0.05 * v_loss

    model.zero_grad(set_to_none=True)
    value_model.zero_grad(set_to_none=True)
    loss.backward()
    policy_grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_norm_clip)
    value_grad_norm = torch.nn.utils.clip_grad_norm_(value_model.parameters(), grad_norm_clip)
    optimizer.step()

    # -------
    # Logging
    # -------

    clipfrac = ((ppo_ratio - 1.0).abs() > 0.2).float().mean().item()
    logger.log("Clip Frac", i, clipfrac)
    logger.log("reward", i, flat_rewards.mean().item())
    logger.log("kl", i, kl.mean().item())
    logger.log("Vf loss", i, v_loss.item())

    # Logging
    if i % 10 == 0:
        print(f"Iter: {i}, Avg reward: {flat_rewards.mean().item():.4f}, KL: {kl.mean().item():.4f}, Policy grad norm: {policy_grad_norm:.3f}, Vf Grad norm: {value_grad_norm:.3f}, clip frac: {clipfrac:.3f}")

    if i % 20 == 0:
        model.eval()
        idx = model.generate(sample_prompt, max_new_tokens=80, top_k=30, do_sample=True).cpu()
        for j,generation in enumerate(idx):
            print(f"Generation {j}:", train_ds.tokenizer.decode(generation))
        model.train()

print("\n===== DONE RL =====")


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!



===== STARTING RL =====
Iter: 0, Avg reward: 0.0475, KL: 0.0000, Policy grad norm: 0.184, Vf Grad norm: 0.043, clip frac: 0.358
Generation 0: ⏎movies no me it`s me thougho down⏎ming after down in Tranking me⏎Ople gud....New
Generation 1: ⏎ i would be sad That suckse⏎ lanks...⏎What it about 2 melt.⏎ that`s gonna go eff
Generation 2: ⏎is sooo having to do..... Busy miss u mum stop my favorite with please kinds⏎den
Iter: 10, Avg reward: 0.0218, KL: 0.0020, Policy grad norm: 0.158, Vf Grad norm: 0.012, clip frac: 0.333
Iter: 20, Avg reward: 0.1968, KL: 0.0081, Policy grad norm: 0.184, Vf Grad norm: 0.011, clip frac: 0.353
Generation 0: ⏎This why hungry to fun my Hayfuse rain I don`t want to get the look like today? 
Generation 1: ⏎Yeah, I am need to be a mom after the perf, Think  Good now take the for my be a
Generation 2: ⏎  whene he do use for this we defun?⏎  http://twitpicAid/twitpic.com/4w142 - Lik
Iter: 30, Avg reward: 0.1311, KL: 0.0072, Policy grad norm: 0.217, Vf Grad norm: 0.01

Run the following cell to generate 3 tweets using the fine-tuned model.

In [11]:
idx = model.generate(sample_prompt, max_new_tokens=128, top_k=30, do_sample=True).cpu()
for j,generation in enumerate(idx):
    print(f"Generation {j}:", train_ds.tokenizer.decode(generation))

Generation 0: ⏎ Yup anything love thank you love myy yael., nen`t happy mother`s Day  happy Mother`s Day to take The Mobing a be Car Ce Five?⏎ 
Generation 1: ⏎i`m i plannio ope .. sorry thanks like you nor i think it`s cholly then the for good there like a BLEATheting Love to like my li
Generation 2: ⏎we hears mading 2  Aind to ouu?  Happy Mother`s Day in Twitte⏎ to love Smaking Good work thanks like a best lame of on the hair 


## Task 1.4
### Evaluaton of MiniGPT Generation after RLHF fine tuning

Examples of the generated text show:

- **Positive words** like "love", "happy", "thanks" appear more frequently.  
- Syntax and spelling are still fragmented -> caused by short pretraining of MiniGPT 
- Coherence is partially present ("Happy Mother`s Day"), but many phrases are disconnected.
**Comparison to pretraining:**
- Before RLHF: mostly neutral, random character sequences, few positive words.  
- After RLHF: the model clearly tends towards **positive sentiment**, showing the intended "happy" biasness.

**Conclusion:** MiniGPT learns to generate more positive tweets via the sentiment reward, even though text quality and coherence remain very limited.



## Task 1.5

## Supervised Fine-Tuning (SFT) and Reward Model in MiniGPT-RLHF

In the current MiniGPT-RLHF setup, **both Supervised Fine-Tuning (SFT) and separate reward model training are skipped**.  

- The model starts directly with a **pretrained MiniGPT**.  
- PPO/RLHF is applied immediately using the `SentimentRewardModel`.  
 

**What SFT and proper reward model training would do:**  
- **SFT:** Train the model on real prompt-response pairs to improve coherence and language quality before RLHF.  
- **Reward Model:** Learn human preference or quality judgments as a rewrad model instead of using a simple heuristic like sentiment.  

**Conclusion:**  
> Currently, the MiniGPT setup skips both SFT and learning a proper reward model. Adding these steps would make RLHF more effective and the generated text more coherent and relevant.
